# Notebook 47 — Normalisation displacement: diagnosis and scope of recalibration

Notebook 46 found that recomputing BatchNorm running statistics on the training partition, with no weight changed,
recovers most of the collapse of the uniformly pruned, fine-tuned CNN (0.271 to 0.502). This notebook establishes what
that means before it is written into the paper.

**Diagnosis (uniform 80%, five seeds).** Four evaluations of the same weights: (1) as saved, running statistics;
(2) batch statistics on the test data itself (train mode, no update); (3) recalibrated by cumulative averaging over
training batches; (4) recalibrated by the same exponential moving average the fine-tune used (momentum 0.1) over 200
training batches. If (2), (3) and (4) agree and exceed (1), the saved statistics are displaced from what any fresh estimate
gives. Per-layer distances between saved and recalibrated running means and variances are recorded.

**Scope.** Recalibration is applied to every recipe and cell with a saved checkpoint: uniform, protected, global, both
gradual schedules, the dense-head cells of Notebook 46, the MLP input-starvation cells (38 / 96 / 192 weights) and the
TON_IoT uniform cell; macro-F1, the number of materially damaged classes and the security rates (benign flagged as attack,
exact attack type) are reported before and after.

**Pre-stated criteria.** (i) Displacement is genuine if evaluations (2), (3) and (4) each exceed (1) by more than 0.10 on
the uniform models. (ii) Recalibration is specific to starved models if it changes protected and global macro-F1 by less
than 0.02 while changing uniform by more than 0.10. (iii) Recalibration repairs the security failure if, after it, the
uniform models' benign-flagged rate is within 5 points of the dense rate and the count of damaged classes falls below 4.
(iv) The MLP and TON collapses are also displacement if recalibration recovers more than half of their loss. GPU runtime.

In [ ]:
# --- Colab bootstrap ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys, copy, json as _json
os.chdir(REPO); sys.path.insert(0, REPO)
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.utils.prune as prune
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from src.config import CFG, PATHS, set_all_seeds
from src.data import load_raw, clean, temporal_within_capture_split
from src import train as TR, models as M, explain as EXP, mitigate
from src.comnet_audit import assign_validation_tiers, calibration_summary, environment_record, write_json
from src.train import load_anchor, predict, per_class_recall_table, feature_columns

assert torch.cuda.is_available(), 'switch to a GPU runtime first'
DEVICE = TR.DEVICE
DATASET = 'ciciot2023'
SEEDS = list(CFG['seeds']); ANCHOR = int(CFG['anchor_seed'])
OUT = PATHS.tables('comnet')
PRACTICAL_LOSS = 0.10
from itertools import combinations
ARCH = 'cnn1d'; KW64 = {'channels': (64, 128)}; KW128 = {'channels': (128, 128)}

KWMLP = {'hidden': (256, 128)}
print('normalisation diagnosis ready')

In [ ]:
df = clean(load_raw(DATASET, subsample=True, seed=ANCHOR), DATASET)
splits = temporal_within_capture_split(df, seed=ANCHOR)
feat_cols = feature_columns(df)
print(f'{len(df):,} rows | {df.label.nunique()} classes')

In [ ]:
# Pruning policies on the CNN. Fine-tune loop identical to src.compression.prune_and_finetune.
def prunable(model):
    return [(mod, 'weight') for mod in model.modules() if isinstance(mod, (nn.Linear, nn.Conv1d))]

def layer_names(model):
    return {mod: n for n, mod in model.named_modules()}

def apply_layer_amounts(model, amounts):
    # amounts: {module_name: sparsity}; every prunable layer must be named (no silent defaults)
    m = copy.deepcopy(model); names = layer_names(m)
    for mod, name in prunable(m):
        a = amounts[names[mod]]
        if a > 0: prune.l1_unstructured(mod, name=name, amount=float(a)); prune.remove(mod, name)
    return m

def layer_sparsity(model):
    names = layer_names(model); out = {}
    for mod, name in prunable(model):
        w = getattr(mod, name); out[names[mod]] = float((w == 0).float().mean())
    z = sum(int((getattr(mod, n) == 0).sum()) for mod, n in prunable(model)); n_ = sum(getattr(mod, n).numel() for mod, n in prunable(model))
    out['prunable_sparsity'] = z / n_; out['remaining_nonzero_prunable'] = n_ - z
    return out

def finetune_masked(model, seed, *, ft_epochs=8, batch_size=4096, lr=5e-4, verbose=False):
    set_all_seeds(seed)
    from sklearn.preprocessing import LabelEncoder, StandardScaler
    le = LabelEncoder().fit(df['label'].to_numpy())
    scaler = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    t = TR.make_tensors(df, splits, feat_cols, le, scaler); Xtr, ytr = t['train']
    model = model.to(DEVICE)
    masks = {(mod, name): (getattr(mod, name) != 0).float() for mod, name in prunable(model)}
    hooks = [getattr(mod, name).register_hook((lambda mk: (lambda g: g * mk))(mk)) for (mod, name), mk in masks.items()]
    w = TR.tempered_class_weights(ytr.numpy(), len(le.classes_))
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch_size, shuffle=True)
    for ep in range(ft_epochs):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        if verbose: print(f'    ft epoch {ep}')
    for h in hooks: h.remove()
    with torch.no_grad():
        for mod, name in prunable(model): getattr(mod, name).mul_((getattr(mod, name) != 0).float())
    return model.eval(), le, scaler

def save_ckpt(model, le, scaler, path):
    torch.save({'state_dict': model.state_dict(), 'classes': list(le.classes_), 'feat_cols': feat_cols,
                'scaler_mean': scaler.mean_, 'scaler_scale': scaler.scale_}, path)


def load_cell(cell, seed, kw):
    m = M.build(ARCH, len(feat_cols), int(df.label.nunique()), **kw).to(DEVICE)
    m.load_state_dict(torch.load(PATHS.model(DATASET, ARCH, cell, seed), map_location=DEVICE, weights_only=False)['state_dict']); return m.eval()
def loss_table(name, rows, m0f1):
    d = pd.DataFrame(rows); g = d.groupby('cell').test_macro_f1.agg(['mean', 'std', 'min', 'max']); g['loss'] = m0f1 - g['mean']; print(f'\n{name}'); print(g.round(4).to_string()); return d, g
print('helpers ready')
from sklearn.preprocessing import LabelEncoder, StandardScaler
def train_tensors():
    le_ = LabelEncoder().fit(df['label'].to_numpy()); sc = StandardScaler().fit(df.loc[splits['train'], feat_cols].to_numpy(np.float32))
    return TR.make_tensors(df, splits, feat_cols, le_, sc)['train'][0]
Xtr_all = train_tensors()
def bn_layers(m): return [mod for mod in m.modules() if isinstance(mod, (nn.BatchNorm1d, nn.BatchNorm2d))]
def recal(model, seed, mode='cumulative', n_batches=200, momentum=0.1):
    m = copy.deepcopy(model)
    for b in bn_layers(m):
        b.reset_running_stats(); b.momentum = None if mode == 'cumulative' else momentum
    m.train(); set_all_seeds(seed); idx = torch.randperm(len(Xtr_all))[: n_batches * 4096]
    with torch.no_grad():
        for i in range(0, len(idx), 4096): m(Xtr_all[idx[i:i + 4096]].to(DEVICE))
    return m.eval()
def predict_batchstats(model, le, scaler, batch_size=4096):
    # train mode (batch statistics) with momentum 0 so running stats are not touched; no dropout in these models.
    # src.train.predict calls model.eval() internally, so the forward pass is done here by hand.
    m = copy.deepcopy(model)
    for b in bn_layers(m): b.momentum = 0.0
    m.train(); sub = df.loc[splits['test']]
    X = torch.tensor(scaler.transform(sub[feat_cols].to_numpy(np.float32)), dtype=torch.float32); outs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size): outs.append(m(X[i:i + batch_size].to(DEVICE)).cpu())
    assert m.training, 'model left train mode'
    return le.transform(sub['label'].to_numpy()), torch.cat(outs).argmax(1).numpy()
def bn_distance(saved, recalibrated):
    out = []
    for i, (a, b) in enumerate(zip(bn_layers(saved), bn_layers(recalibrated))):
        rm = float((a.running_mean - b.running_mean).norm() / (b.running_mean.norm() + 1e-8)); rv = float((a.running_var - b.running_var).norm() / (b.running_var.norm() + 1e-8))
        out.append({'bn_layer': i, 'rel_mean_distance': rm, 'rel_var_distance': rv, 'saved_var_median': float(a.running_var.median()), 'recal_var_median': float(b.running_var.median())})
    return out
print('helpers ready; training tensor', tuple(Xtr_all.shape))

In [ ]:
# ---------------- Diagnosis on the uniform 80% CNN models ----------------
rowsD, dist = [], []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor(DATASET, ARCH, 'M0_paired', seed, arch_kwargs=KW64); mp = load_cell('prune80_paired', seed, KW64)
    yt, p1, _ = predict(mp, df, splits, le, scaler, feat_cols, which='test'); f_saved = f1_score(yt, p1, average='macro')
    yt, p2 = predict_batchstats(mp, le, scaler); f_batch = f1_score(yt, p2, average='macro')
    mc = recal(mp, seed, 'cumulative'); yt, p3, _ = predict(mc, df, splits, le, scaler, feat_cols, which='test'); f_cum = f1_score(yt, p3, average='macro')
    me = recal(mp, seed, 'ema'); yt, p4, _ = predict(me, df, splits, le, scaler, feat_cols, which='test'); f_ema = f1_score(yt, p4, average='macro')
    rowsD.append({'seed': seed, 'saved_running_stats': f_saved, 'batch_stats_on_test': f_batch, 'recal_cumulative': f_cum, 'recal_ema_0.1': f_ema})
    for d in bn_distance(mp, mc): dist.append({'seed': seed, **d})
    print(f'  seed {seed}: saved {f_saved:.4f} | batch-stats {f_batch:.4f} | recal cumulative {f_cum:.4f} | recal EMA {f_ema:.4f}')
dD = pd.DataFrame(rowsD); dD.to_csv(OUT / 'recal_diagnosis_uniform.csv', index=False); pd.DataFrame(dist).to_csv(OUT / 'recal_bn_distances_uniform.csv', index=False)
print('\nmeans:'); print(dD.drop(columns='seed').mean().round(4).to_string())
print('\nper-layer relative distance saved vs recalibrated (mean over seeds):'); print(pd.DataFrame(dist).groupby('bn_layer')[['rel_mean_distance', 'rel_var_distance', 'saved_var_median', 'recal_var_median']].mean().round(4).to_string())
m = dD.drop(columns='seed').mean(); crit_i = bool(all(m[k] - m['saved_running_stats'] > 0.10 for k in ['batch_stats_on_test', 'recal_cumulative', 'recal_ema_0.1']))
print('\n(i) displacement genuine (all three fresh estimates exceed saved by > 0.10):', crit_i)

In [ ]:
# ---------------- Scope: recalibration across recipes, with per-class damage and security rates ----------------
fam_map = pd.read_csv(OUT / 'ciciot2023_alert_family_mapping.csv').set_index('fine_label')['alert_family'].to_dict()
CELLS = [('cnn1d', 'uniform80', 'prune80_paired', KW64, 'M0_paired'), ('cnn1d', 'protected', 'layerwise80_protect_conv0_paired', KW64, 'M0_paired'), ('cnn1d', 'global80', 'global80_paired', KW64, 'M0_paired'),
         ('cnn1d', 'gradual_perlayer', 'gradual_perlayer80_paired', KW64, 'M0_paired'), ('cnn1d', 'gradual_global', 'gradual_global80_paired', KW64, 'M0_paired'),
         ('cnn1d', 'starved_densehead', 'ctrl_starved_densehead_paired', KW64, 'M0_paired'), ('cnn1d', 'f128_uniform80', 'f128_conv0dose800_paired', KW128, 'M0_f128_paired'),
         ('mlp', 'mlp_38w', 'inputdose9962_mlp_paired', KWMLP, 'M0'), ('mlp', 'mlp_96w', 'inputdose9904_mlp_paired', KWMLP, 'M0'), ('mlp', 'mlp_192w', 'inputdose9808_mlp_paired', KWMLP, 'M0')]
def sec_rates(yt, yp, le):
    classes = list(le.classes_); b = classes.index('BenignTraffic'); yt = np.asarray(yt); yp = np.asarray(yp)
    return {'benign_to_attack': float((yp[yt == b] != b).mean()), 'exact_attack_type': float((yp[yt != b] == yt[yt != b]).mean())}
rows, per_class = [], []
for arch, name, cell, kw, base in CELLS:
    for seed in SEEDS:
        m0, le, scaler, _ = load_anchor(DATASET, arch, base, seed, arch_kwargs=kw)
        m = M.build(arch, len(feat_cols), len(le.classes_), **kw).to(DEVICE); m.load_state_dict(torch.load(PATHS.model(DATASET, arch, cell, seed), map_location=DEVICE, weights_only=False)['state_dict']); m.eval()
        yv, pv, _ = predict(m0, df, splits, le, scaler, feat_cols, which='val'); yt0, pt0, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test')
        yt, pb, _ = predict(m, df, splits, le, scaler, feat_cols, which='test'); mr = recal(m, seed, 'cumulative'); yt, pa, _ = predict(mr, df, splits, le, scaler, feat_cols, which='test')
        f0, fb, fa = f1_score(yt0, pt0, average='macro'), f1_score(yt, pb, average='macro'), f1_score(yt, pa, average='macro')
        rows.append({'arch': arch, 'cell': name, 'seed': seed, 'M0': f0, 'before': fb, 'after': fa, 'loss_before': f0 - fb, 'loss_after': f0 - fa, **{k + '_before': v for k, v in sec_rates(yt, pb, le).items()}, **{k + '_after': v for k, v in sec_rates(yt, pa, le).items()}, **{k + '_dense': v for k, v in sec_rates(yt0, pt0, le).items()}})
        r0 = per_class_recall_table(yt0, pt0, le).set_index('label')['recall']; rv = per_class_recall_table(yv, pv, le).set_index('label')['recall']
        for tag, pp in (('before', pb), ('after', pa)):
            rc = per_class_recall_table(yt, pp, le).set_index('label')['recall']
            for cls in r0.index: per_class.append({'arch': arch, 'cell': name, 'seed': seed, 'stage': tag, 'class': cls, 'recall_loss': float(r0.loc[cls] - rc.loc[cls]), 'val_recall': float(rv.loc[cls])})
    print(f'  {name}: done')
R = pd.DataFrame(rows); R.to_csv(OUT / 'recal_scope_wide.csv', index=False); PC = pd.DataFrame(per_class); PC.to_csv(OUT / 'recal_scope_per_class_effects.csv', index=False)
# damaged-class count uses the same rule as the paper: loss >= 0.10 and beyond the class's validation 2-sd band, in >= 3/5 seeds
def damaged(arch, name, stage):
    sub = PC[(PC.arch == arch) & (PC.cell == name) & (PC.stage == stage)]
    band = 2 * sub.groupby('class').val_recall.std(); mat = sub.assign(mat=lambda d: (d.recall_loss >= 0.10) & (d.recall_loss > d['class'].map(band)))
    return int((mat.groupby('class').mat.mean() >= 0.6).sum())
summ = R.groupby(['arch', 'cell']).agg(M0=('M0', 'mean'), before=('before', 'mean'), after=('after', 'mean'), loss_before=('loss_before', 'mean'), loss_after=('loss_after', 'mean'),
        benign_to_attack_before=('benign_to_attack_before', 'mean'), benign_to_attack_after=('benign_to_attack_after', 'mean'), benign_to_attack_dense=('benign_to_attack_dense', 'mean'),
        exact_attack_before=('exact_attack_type_before', 'mean'), exact_attack_after=('exact_attack_type_after', 'mean')).reset_index()
summ['damaged_before'] = [damaged(a, c, 'before') for a, c in zip(summ.arch, summ.cell)]; summ['damaged_after'] = [damaged(a, c, 'after') for a, c in zip(summ.arch, summ.cell)]
summ['recovered_fraction'] = (summ.loss_before - summ.loss_after) / summ.loss_before.clip(lower=1e-6); summ.to_csv(OUT / 'recal_scope_summary.csv', index=False)
pd.set_option('display.width', 250); print(summ.round(4).to_string(index=False))

In [ ]:
# ---------------- TON_IoT uniform cell ----------------
from src.data import temporal_within_capture_split as ton_split
df_c, splits_c, feat_c, Xtr_c = df, splits, feat_cols, Xtr_all
df = clean(load_raw('ton_iot', subsample=True, seed=ANCHOR), 'ton_iot'); splits = ton_split(df, seed=ANCHOR); feat_cols = feature_columns(df); Xtr_all = train_tensors()
rowsT = []
for seed in SEEDS:
    m0, le, scaler, _ = load_anchor('ton_iot', 'cnn1d', 'M0_paired', seed, arch_kwargs=KW64)
    m = M.build('cnn1d', len(feat_cols), len(le.classes_), **KW64).to(DEVICE); m.load_state_dict(torch.load(PATHS.model('ton_iot', 'cnn1d', 'conv0dose80_paired', seed), map_location=DEVICE, weights_only=False)['state_dict']); m.eval()
    yt0, pt0, _ = predict(m0, df, splits, le, scaler, feat_cols, which='test'); yt, pb, _ = predict(m, df, splits, le, scaler, feat_cols, which='test'); mr = recal(m, seed, 'cumulative'); yt, pa, _ = predict(mr, df, splits, le, scaler, feat_cols, which='test')
    rowsT.append({'seed': seed, 'M0': f1_score(yt0, pt0, average='macro'), 'before': f1_score(yt, pb, average='macro'), 'after': f1_score(yt, pa, average='macro')})
T = pd.DataFrame(rowsT); T['loss_before'] = T.M0 - T.before; T['loss_after'] = T.M0 - T.after; T.to_csv(OUT / 'recal_ton_uniform.csv', index=False)
print(T.round(4).to_string(index=False)); print(f'\nTON uniform: loss {T.loss_before.mean():.3f} -> {T.loss_after.mean():.3f} (recovered {(T.loss_before.mean()-T.loss_after.mean())/T.loss_before.mean():.0%})')
df, splits, feat_cols, Xtr_all = df_c, splits_c, feat_c, Xtr_c

In [ ]:
# ---------------- Verdict + commit ----------------
S = summ.set_index('cell')
verdict = pd.DataFrame([
 {'criterion': 'i_displacement_genuine_three_fresh_estimates_exceed_saved_by_0.10', 'value': str({k: round(float(v), 3) for k, v in dD.drop(columns='seed').mean().items()}), 'pass': crit_i},
 {'criterion': 'ii_specific_to_starved_uniform_change_gt_0.10_protected_and_global_lt_0.02', 'value': f"uniform {S.loc['uniform80','after']-S.loc['uniform80','before']:+.3f}, protected {S.loc['protected','after']-S.loc['protected','before']:+.3f}, global {S.loc['global80','after']-S.loc['global80','before']:+.3f}", 'pass': bool(S.loc['uniform80','after']-S.loc['uniform80','before'] > 0.10 and abs(S.loc['protected','after']-S.loc['protected','before']) < 0.02 and abs(S.loc['global80','after']-S.loc['global80','before']) < 0.02)},
 {'criterion': 'iii_repairs_security_benign_within_5pts_of_dense_and_damaged_lt_4', 'value': f"benign {S.loc['uniform80','benign_to_attack_after']:.3f} vs dense {S.loc['uniform80','benign_to_attack_dense']:.3f}; damaged {int(S.loc['uniform80','damaged_after'])}", 'pass': bool(abs(S.loc['uniform80','benign_to_attack_after']-S.loc['uniform80','benign_to_attack_dense']) < 0.05 and S.loc['uniform80','damaged_after'] < 4)},
 {'criterion': 'iv_mlp38_and_ton_recover_gt_half', 'value': f"mlp38 {S.loc['mlp_38w','recovered_fraction']:.2f}, ton {(T.loss_before.mean()-T.loss_after.mean())/T.loss_before.mean():.2f}", 'pass': bool(S.loc['mlp_38w','recovered_fraction'] > 0.5 and (T.loss_before.mean()-T.loss_after.mean())/T.loss_before.mean() > 0.5)},
]); print(verdict.to_string(index=False)); verdict.to_csv(OUT / 'recal_gate_verdict.csv', index=False)
write_json(OUT / 'recal_environment.json', {'seeds': SEEDS, 'n_batches': 200, 'environment': environment_record()})
import subprocess, shutil, glob
_b = subprocess.run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], capture_output=True, text=True).stdout.strip(); assert _b == 'main', f'checked-out branch is {_b!r}'
subprocess.run(['git', 'config', '--global', 'user.name', 'Md Anas Biswas'], check=True); subprocess.run(['git', 'config', '--global', 'user.email', 'anasbiswas@gmail.com'], check=True)
cred = '/content/drive/MyDrive/IoT_Trust_Research/.git-credentials'
if os.path.exists(cred): shutil.copy(cred, '/root/.git-credentials'); subprocess.run(['git', 'config', '--global', 'credential.helper', 'store'], check=True)
_own = 'notebooks/47_normalisation_displacement.ipynb'
if os.path.exists(_own):
    d_ = _json.load(open(_own))
    for c in d_.get('cells', []):
        if c.get('cell_type') == 'code': c['outputs'] = []; c['execution_count'] = None
    _json.dump(d_, open(_own, 'w'), indent=1)
subprocess.run(['git', 'add', _own] + glob.glob('results/tables/comnet/recal_*'), check=True)
r = subprocess.run(['git', 'commit', '-m', 'notebook 47: normalisation displacement - diagnosis on uniform models, recalibration across recipes/architectures/datasets with per-class and security rates'], capture_output=True, text=True); print(r.stdout or r.stderr)
print(subprocess.run(['git', 'push'], capture_output=True, text=True).stderr or 'pushed'); print(subprocess.run(['git', 'log', '--oneline', '-2'], capture_output=True, text=True).stdout)